In [ ]:
!git clone https://github.com/hubarruby/MaskPure.git
%cd MaskPure

In [ ]:
# --- Step 0: Make sure GPU is enabled ---
# Runtime -> Change runtime type -> GPU
import torch
print("GPU Available:", torch.cuda.is_available())

In [ ]:
!sudo apt-get install python3.11 python3.11-venv python3.11-dev
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 2
!sudo update-alternatives --config python3
!python --version

In [ ]:
# Install ensurepip for Python 3.11
!python3.11 -m ensurepip --upgrade

# Upgrade pip for Python 3.11
!python3.11 -m pip install --upgrade pip setuptools wheel

# Check versions
!python3.11 -m pip --version

In [ ]:
# PyTorch CPU / GPU (adjust as needed)
!python3.11 -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Transformers + datasets
!python3.11 -m pip install transformers
!python3.11 -m pip install datasets==4.6.1

# OpenAttack (Python 3.11 compatible)
!python3.11 -m pip install OpenAttack



In [ ]:
!mv /content/snippet.py /content/MaskPure/

In [ ]:
!python3.11 /content/MaskPure/snippet.py

--- Diagnostic from snippet.py ---
OpenAttack module file (from snippet.py): /usr/local/lib/python3.11/dist-packages/OpenAttack/__init__.py
Python sys.path (from snippet.py):
  /content/MaskPure
  /env/python
  /usr/lib/python311.zip
  /usr/lib/python3.11
  /usr/lib/python3.11/lib-dynload
  /usr/local/lib/python3.11/dist-packages
  /usr/lib/python3/dist-packages
  /usr/local/lib/python3.11/dist-packages/setuptools/_vendor
  /tmp/tmpt6pkhn_f
NLTK data path (from snippet.py): ['/content/MaskPure/data', '/root/nltk_data', '/usr/nltk_data', '/usr/share/nltk_data', '/usr/lib/nltk_data', '/usr/share/nltk_data', '/usr/local/share/nltk_data', '/usr/lib/nltk_data', '/usr/local/lib/nltk_data']
Does /content/MaskPure/data/TProcess.NLTKSentTokenizer/english.pickle exist?: True
Does /root/nltk_data/TProcess.NLTKSentTokenizer/english.pickle exist?: True
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordn

In [ ]:
import sys
import OpenAttack
import os

print(f"OpenAttack module file: {OpenAttack.__file__}")
print("Python sys.path:")
for p in sys.path:
    print(f"  {p}")

# Also check if the problematic path exists
problematic_path = '/content/MaskPure/data/TProcess.NLTKSentTokenizer/english.pickle'
print(f"Does problematic path '{problematic_path}' exist? {os.path.exists(problematic_path)}")


In [ ]:
import sys
import site
import os

# Find the site-packages directory for python3.11
python311_site_packages = None
# site.getsitepackages() returns a list of directories where Python packages are installed.
# We iterate through them to find the one that pertains to python3.11
for sp in site.getsitepackages():
    if 'python3.11' in sp:
        python311_site_packages = sp
        break

# If found and not already in sys.path, add it.
if python311_site_packages and python311_site_packages not in sys.path:
    sys.path.insert(0, python311_site_packages)
    print(f"Added {python311_site_packages} to sys.path")
else:
    print(f"Could not find python3.11 site-packages or it's already in sys.path: {python311_site_packages}")

# Verify that OpenAttack can now be imported by the kernel
try:
    import OpenAttack
    print(f"OpenAttack module found by kernel at: {OpenAttack.__file__}")
except ModuleNotFoundError:
    print("ModuleNotFoundError: OpenAttack still not found in kernel environment after path modification.")


In [ ]:
import os
import nltk
import sys

# Path to the snippet.py file
snippet_file_path = "/content/MaskPure/snippet.py"

# Read the original content of snippet.py
with open(snippet_file_path, 'r') as f:
    snippet_content_lines = f.readlines()

# Prepare the lines to prepend for NLTK data path modification
# and diagnostic information
prefix_lines_snippet = [
    "import os\n",
    "import nltk\n",
    "import sys\n",
    "import OpenAttack\n", # Ensure OpenAttack is imported first
    "\n",
    "print(f\"--- Diagnostic from snippet.py ---\")\n",
    "print(f\"OpenAttack module file (from snippet.py): {OpenAttack.__file__}\")\n",
    "print(\"Python sys.path (from snippet.py):\")\n",
    "for p in sys.path:\n",
    "    print(f\"  {p}\")\n",
    "\n",
    "# Add the parent directory of the NLTK data used by OpenAttack to NLTK's data path.\n",
    "script_dir = os.path.dirname(os.path.abspath(__file__))\n",
    "nltk_data_path_to_add = os.path.join(script_dir, \"data\")\n",
    "if nltk_data_path_to_add not in nltk.data.path:\n",
    "    nltk.data.path.insert(0, nltk_data_path_to_add)\n",
    "print(\"NLTK data path (from snippet.py):\", nltk.data.path)\n",
    "print(f\"Does /content/MaskPure/data/TProcess.NLTKSentTokenizer/english.pickle exist?: {os.path.exists('/content/MaskPure/data/TProcess.NLTKSentTokenizer/english.pickle')}\")\n",
    "print(f\"Does /root/nltk_data/TProcess.NLTKSentTokenizer/english.pickle exist?: {os.path.exists('/root/nltk_data/TProcess.NLTKSentTokenizer/english.pickle')}\")\n",
    "\n" # Add a final newline for separation
]

# Remove any existing 'import os', 'import nltk', 'import sys', 'import OpenAttack'
# and previous NLTK path modification/diagnostic lines from the original content
new_snippet_content_lines = []
for line in snippet_content_lines:
    stripped_line = line.strip()
    if not (stripped_line.startswith("import os") or
            stripped_line.startswith("import nltk") or
            stripped_line.startswith("import sys") or
            stripped_line.startswith("import OpenAttack") or
            "# Add the parent directory of the NLTK data used by OpenAttack to NLTK's data path." in line or
            "script_dir = os.path.dirname(os.path.abspath(__file__))" in line or
            "nltk_data_path_to_add = os.path.join(script_dir, \"data\")" in line or
            "if nltk_data_path_to_add not in nltk.data.path:" in line or
            "nltk.data.path.insert(0, nltk_data_path_to_add)" in line or
            ("print(" in line and "from snippet.py" in line)): # Catch various print statements
        new_snippet_content_lines.append(line)

# Write the modified content back to snippet.py
with open(snippet_file_path, 'w') as f:
    f.writelines(prefix_lines_snippet)
    f.writelines(new_snippet_content_lines)

print(f"Successfully patched {snippet_file_path} with diagnostics.")

Successfully patched /content/MaskPure/snippet.py with diagnostics.


I've added extensive diagnostic prints to `snippet.py` to help us understand why the 'Unsafe resource path' error is occurring. Please execute `snippet.py` again by running the cell with `!python3.11 snippet.py`.

In [ ]:
import sys
import os

print(f"Current Python executable: {sys.executable}")
print(f"Current Python version: {sys.version}")
print("Current sys.path:")
for p in sys.path:
    print(f"  {p}")

try:
    import OpenAttack
    print(f"OpenAttack module found at: {OpenAttack.__file__}")
except ModuleNotFoundError:
    print("ModuleNotFoundError: OpenAttack still not found in kernel environment.")

# Check pip list for OpenAttack for python3.11
print("\nRunning pip list for python3.11:")
!python3.11 -m pip list | grep OpenAttack


Current Python executable: /usr/bin/python3
Current Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Current sys.path:
  /content
  /env/python
  /usr/lib/python312.zip
  /usr/lib/python3.12
  /usr/lib/python3.12/lib-dynload
  
  /usr/local/lib/python3.12/dist-packages
  /usr/lib/python3/dist-packages
  /usr/local/lib/python3.12/dist-packages/IPython/extensions
  /root/.ipython
ModuleNotFoundError: OpenAttack still not found in kernel environment.

Running pip list for python3.11:
OpenAttack               2.1.1


Please execute the above cell and share the output. This will help us diagnose why `OpenAttack` isn't found by the kernel.

Please execute the above cell to provide more diagnostic information about `OpenAttack`'s environment.

In [ ]:
# Copy the OpenAttack-specific NLTK data to a trusted NLTK location
!cp -r /content/MaskPure/data/TProcess.NLTKSentTokenizer /root/nltk_data/

print("Copied TProcess.NLTKSentTokenizer data to /root/nltk_data/")

In [ ]:
import os
import nltk

# Path to the snippet.py file
snippet_file_path = "/content/MaskPure/snippet.py"

# Read the original content of snippet.py
with open(snippet_file_path, 'r') as f:
    snippet_content_lines = f.readlines()

# Prepare the clean prefix lines
prefix_lines_snippet = [
    "import os\n",
    "import nltk\n",
    "\n" # Add a final newline for separation
]

# Filter out all lines related to the problematic NLTK path modification and duplicate imports
new_snippet_content_lines = []
for line in snippet_content_lines:
    stripped_line = line.strip()
    if not (stripped_line.startswith("import os") or
            stripped_line.startswith("import nltk") or
            "# Add the parent directory of the NLTK data used by OpenAttack to NLTK's data path." in line or
            "script_dir = os.path.dirname(os.path.abspath(__file__))" in line or
            "nltk_data_path_to_add = os.path.join(script_dir, \"data\")" in line or
            "if nltk_data_path_to_add not in nltk.data.path:" in line or
            "nltk.data.path.insert(0, nltk_data_path_to_add)" in line or
            "print(\"NLTK data path in snippet.py:\"" in line):
        new_snippet_content_lines.append(line)

# Write the modified content back to snippet.py
with open(snippet_file_path, 'w') as f:
    f.writelines(prefix_lines_snippet)
    f.writelines(new_snippet_content_lines)

print(f"Successfully cleaned and re-patched {snippet_file_path}")

In [ ]:
import os
import nltk

# Path to the maskpure_sst2_openattack.py file
maskpure_file_path = "/content/MaskPure/maskpure_sst2_openattack.py"

# Read the original content of maskpure_file_path
with open(maskpure_file_path, 'r') as f:
    maskpure_content_lines = f.readlines()

# Prepare the clean prefix lines
prefix_lines_maskpure = [
    "import os\n",
    "import nltk\n",
    "\n" # Add a final newline for separation
]

# Filter out all lines related to the problematic NLTK path modification and duplicate imports
new_maskpure_content_lines = []
for line in maskpure_content_lines:
    stripped_line = line.strip()
    if not (stripped_line.startswith("import os") or
            stripped_line.startswith("import nltk") or
            "# Add the parent directory of the NLTK data used by OpenAttack to NLTK's data path." in line or
            "script_dir = os.path.dirname(os.path.abspath(__file__))" in line or
            "nltk_data_path_to_add = os.path.join(script_dir, \"data\")" in line or
            "if nltk_data_path_to_add not in nltk.data.path:" in line or
            "nltk.data.path.insert(0, nltk_data_path_to_add)" in line or
            "print(\"NLTK data path in snippet.py:\"" in line): # Assuming same print statement was used if any
        new_maskpure_content_lines.append(line)

# Write the modified content back to maskpure_sst2_openattack.py
with open(maskpure_file_path, 'w') as f:
    f.writelines(prefix_lines_maskpure)
    f.writelines(new_maskpure_content_lines)

print(f"Successfully cleaned and re-patched {maskpure_file_path}")

I've copied the NLTK-related data to a trusted location and re-patched both scripts to remove the debugging print. Please re-run the cells that execute `snippet.py` and `maskpure_sst2_openattack.py` again to see if the error is resolved.

In [ ]:
import os
import nltk

# Path to the snippet.py file
snippet_file_path = "/content/MaskPure/snippet.py"

# Read the original content of snippet.py
with open(snippet_file_path, 'r') as f:
    snippet_content_lines = f.readlines()

# Prepare the lines to prepend for NLTK data path modification
prefix_lines_snippet = [
    "import os\n",
    "import nltk\n",
    "\n", # Add a newline for separation
    "# Add the parent directory of the NLTK data used by OpenAttack to NLTK's data path.\n",
    "# The data is expected at ./data/TProcess.NLTKSentTokenizer/english.pickle.\n",
    "# So, the directory ./data needs to be added to nltk.data.path.\n",
    "script_dir = os.path.dirname(os.path.abspath(__file__))\n",
    "nltk_data_path_to_add = os.path.join(script_dir, \"data\")\n",
    "if nltk_data_path_to_add not in nltk.data.path:\n",
    "    nltk.data.path.insert(0, nltk_data_path_to_add)\n",
    "print(\"NLTK data path in snippet.py:\", nltk.data.path)\n", # Added print statement for debugging
    "\n" # Add a final newline for separation
]

# Remove any existing 'import os' or 'import nltk' from the original content to avoid duplicates
new_snippet_content_lines = []
for line in snippet_content_lines:
    if "import os" not in line.strip() and "import nltk" not in line.strip():
        new_snippet_content_lines.append(line)

# Write the modified content back to snippet.py
with open(snippet_file_path, 'w') as f:
    f.writelines(prefix_lines_snippet)
    f.writelines(new_snippet_content_lines)

print(f"Successfully patched {snippet_file_path}")

In [ ]:
import os
import nltk

# Path to the maskpure_sst2_openattack.py file
maskpure_file_path = "/content/MaskPure/maskpure_sst2_openattack.py"

# Read the original content of maskpure_sst2_openattack.py
with open(maskpure_file_path, 'r') as f:
    maskpure_content_lines = f.readlines()

# Prepare the lines to prepend for NLTK data path modification
prefix_lines_maskpure = [
    "import os\n",
    "import nltk\n",
    "\n", # Add a newline for separation
    "# Add the parent directory of the NLTK data used by OpenAttack to NLTK's data path.\n",
    "# The data is expected at ./data/TProcess.NLTKSentTokenizer/english.pickle.\n",
    "# So, the directory ./data needs to be added to nltk.data.path.\n",
    "script_dir = os.path.dirname(os.path.abspath(__file__))\n",
    "nltk_data_path_to_add = os.path.join(script_dir, \"data\")\n",
    "if nltk_data_path_to_add not in nltk.data.path:\n",
    "    nltk.data.path.insert(0, nltk_data_path_to_add)\n",
    "\n" # Add a final newline for separation
]

# Remove any existing 'import os' or 'import nltk' from the original content to avoid duplicates
new_maskpure_content_lines = []
for line in maskpure_content_lines:
    if "import os" not in line.strip() and "import nltk" not in line.strip():
        new_maskpure_content_lines.append(line)

# Write the modified content back to maskpure_sst2_openattack.py
with open(maskpure_file_path, 'w') as f:
    f.writelines(prefix_lines_maskpure)
    f.writelines(new_maskpure_content_lines)

print(f"Successfully patched {maskpure_file_path}")

I've applied the patch to both `snippet.py` and `maskpure_sst2_openattack.py`. Now, please try running the cells that execute these scripts again.

In [ ]:
!mv /content/maskpure_sst2_openattack.py /content/MaskPure/

In [ ]:
!python3.11 /content/MaskPure/maskpure_sst2_openattack.py

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
Using device: cuda
Loading weights: 100% 199/199 [00:00<00:00, 1132.92it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 